# 02 — Preprocesamiento y Train/Test Split

**TFM — Valoración Inmobiliaria Masiva: SLX vs Machine Learning**  
Dataset: King County House Prices (21.613 viviendas, 2014–2015)

---

## ¿Para qué sirve este notebook?

El EDA (notebook 01) identificó los problemas del dataset crudo y las transformaciones necesarias.  
Este notebook las implementa de forma ordenada y produce los artefactos que consumen los notebooks posteriores:

- `data/processed/kc_clean.csv` — Dataset limpio completo
- `data/processed/X_train.csv` / `X_test.csv` — Features de entrenamiento y test
- `data/processed/y_train.csv` / `y_test.csv` — Target log(price)
- `data/processed/coords_train.csv` / `coords_test.csv` — Coordenadas lat/long para notebook 03

---

### Transformaciones aplicadas
1. Eliminación de `id` (identificador sin poder predictivo)
2. Ingeniería temporal: `date` → `sale_month` + `house_age = sale_year − yr_built`
3. Binarización: `yr_renovated` → `renovated` (0/1)
4. Eliminación del outlier `bedrooms = 33` (error de entrada)
5. Eliminación de `sqft_basement` (colinealidad perfecta con `sqft_living − sqft_above`)
6. Log-transformación del precio: `log_price = log(price)`
7. Train/Test split 80/20 (SEED = 42)
8. Guardado de artefactos en `data/processed/`

---
## Setup y carga de datos

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

RAW_PATH  = '../data/raw/kc_house_data.csv'
PROC_PATH = '../data/processed/'
os.makedirs(PROC_PATH, exist_ok=True)

SEED = 42

df = pd.read_csv(RAW_PATH)
print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'Columnas: {list(df.columns)}')

---
## 1. Eliminación de `id`

`id` es el identificador único de cada transacción. No contiene información sobre las características
físicas ni la localización del inmueble, por lo que no aporta ningún poder predictivo.
Mantenerlo en el modelo podría inducir sobreajuste espurio si algún algoritmo le asignara relevancia.

In [ ]:
df = df.drop(columns=['id'])
print(f'Tras eliminar id: {df.shape[1]} columnas')

---
## 2. Ingeniería de variables temporales

### `sale_month` — estacionalidad del mercado
El EDA mostró que las ventas no se distribuyen uniformemente a lo largo del año: la primavera
concentra más transacciones y precios ligeramente superiores. Extraemos el mes de venta como
variable categórica ordinal para que los modelos puedan capturar este efecto estacional.

### `house_age` — antigüedad relativa
`yr_built` en valor absoluto mezcla distintas épocas constructivas de forma opaca.
Transformarlo en `house_age = año_venta − yr_built` expresa la antigüedad en términos
relativos al momento de la transacción, que es lo que económicamente importa al comprador.
Además, esta representación sería portable a datasets de otros años sin recodificación.

Se descartan `date`, `date_parsed`, `yr_built` y `sale_year` tras extraer la información relevante.

In [ ]:
df['date_parsed'] = pd.to_datetime(df['date'], format='%Y%m%dT%H%M%S')
df['sale_month']  = df['date_parsed'].dt.month
df['sale_year']   = df['date_parsed'].dt.year
df['house_age']   = df['sale_year'] - df['yr_built']

df = df.drop(columns=['date', 'date_parsed', 'yr_built', 'sale_year'])

print(f'sale_month — rango: {df["sale_month"].min()}–{df["sale_month"].max()}')
print(f'house_age  — rango: {df["house_age"].min()}–{df["house_age"].max()} años')
print(f'Columnas restantes: {df.shape[1]}')

---
## 3. Binarización de `yr_renovated` → `renovated`

El 95,8% de los registros tiene `yr_renovated = 0` (código especial para «no renovada»).
Tratar esta variable como numérica continua es incorrecto: el año de renovación no tiene
una relación monótona clara con el precio (una renovación de 1990 no es necesariamente
mejor que una de 2005 en términos de impacto sobre el precio actual).

El EDA confirmó que el hecho binario de haber sido renovada sí discrimina significativamente
el precio: t-test con **p = 1,43 × 10⁻⁶³**. Se crea la variable `renovated` (0 = no renovada,
1 = renovada) y se descarta `yr_renovated`.

In [ ]:
df['renovated'] = (df['yr_renovated'] > 0).astype(int)
df = df.drop(columns=['yr_renovated'])

n_renov = df['renovated'].sum()
print(f'Viviendas renovadas : {n_renov:,} ({n_renov/len(df)*100:.1f}%)')
print(f'No renovadas        : {len(df)-n_renov:,} ({(len(df)-n_renov)/len(df)*100:.1f}%)')

---
## 4. Eliminación del outlier `bedrooms = 33`

El EDA detectó una vivienda con 33 dormitorios y solo 1.620 ft² de superficie habitable,
lo que equivale a habitaciones de **4,6 m²** — físicamente imposible.
El precio (640.000 $) también es inconsistente con 33 dormitorios.
Se trata de un error de entrada de datos (posiblemente son 3 habitaciones).

Mantener este registro sesgaría cualquier modelo que aprenda de `bedrooms`,
especialmente en la interacción con `sqft_living`.

In [ ]:
n_antes = len(df)
df = df[df['bedrooms'] != 33].reset_index(drop=True)
n_despues = len(df)

print(f'Registros eliminados: {n_antes - n_despues}')
print(f'Tamaño del dataset  : {n_despues:,} viviendas')
print(f'Máximo bedrooms ahora: {df["bedrooms"].max()}')

---
## 5. Eliminación de `sqft_basement` (variable redundante)

El EDA demostró que `sqft_basement = sqft_living − sqft_above` en el **100%** de los registros:
es una variable algebraicamente derivable, no una medición independiente.

**Implicaciones por tipo de modelo:**
- **SLX (regresión lineal):** genera colinealidad perfecta → la matriz X'X es singular y no invertible.  
  El modelo no puede estimar coeficientes de forma única.
- **Random Forest / XGBoost:** aunque los árboles toleran la multicolinealidad, la variable es
  redundante y compite con `sqft_living` y `sqft_above` por los splits, diluyendo su importancia
  individual sin añadir información nueva.

Se elimina para evitar ambos problemas.

In [ ]:
# Verificación antes de eliminar
diff = (df['sqft_living'] - df['sqft_above'] - df['sqft_basement']).abs()
print(f'sqft_living = sqft_above + sqft_basement en {(diff == 0).mean()*100:.1f}% de los casos')

df = df.drop(columns=['sqft_basement'])
print(f'Columnas restantes: {df.shape[1]}')

---
## 6. Log-transformación del precio: `log_price = log(price)`

### Justificación estadística
El precio de la vivienda sigue empíricamente una distribución **log-normal**: la mayoría de
las viviendas se concentran en precios medios, pero una cola larga de inmuebles de lujo
genera un sesgo (*skewness*) de 4,02. Los modelos de regresión minimizadores del MSE
funcionan mejor cuando el target es aproximadamente simétrico.

Tras aplicar `log(price)` el sesgo cae a **0,43**, aproximando la distribución a la normal.

### Justificación práctica
- El error de predicción sobre `log(price)` es equivalente al **error porcentual** sobre el precio
  original (propiedad de la transformación logarítmica), que es la métrica natural del sector
  inmobiliario (p. ej.: «error del 5%» vs «error de 25.000 $»).
- Reduce la influencia de las mansiones de lujo sin necesidad de eliminarlas.
- Mejora la homocedasticidad de los residuos en el modelo SLX.

Se usa `np.log` (logaritmo natural) y no `log1p` dado que `price > 0` siempre,
y es consistente con el cálculo del Índice de Moran del notebook 01.
Se conserva `price` original para poder calcular el RMSE en escala de dólares al evaluar.

In [ ]:
df['log_price'] = np.log(df['price'])

print(f'Skewness price     : {df["price"].skew():.3f}')
print(f'Skewness log_price : {df["log_price"].skew():.3f}')
print(f'Rango log_price    : {df["log_price"].min():.2f} – {df["log_price"].max():.2f}')

---
## 7. Dataset final — lista de variables

Tras todas las transformaciones, el dataset queda con **18 features predictoras** + 2 columnas
de precio (`price` y `log_price`). Se excluyen `price` y `log_price` de la matriz X:
`log_price` es el target y `price` se reserva para la evaluación en escala original.

**Las coordenadas `lat` y `long` se mantienen en X** porque el notebook 03 las necesita
para calcular los retardos espaciales KNN. En el modelo SLX se excluirán explícitamente
ya que la estructura espacial entra por la matriz de pesos W.

In [ ]:
FEATURES = [c for c in df.columns if c not in ['price', 'log_price']]

print(f'Dataset final: {df.shape[0]:,} filas × {len(FEATURES)} features + target')
print(f'\nFeatures ({len(FEATURES)}):')
for f in FEATURES:
    print(f'  {f}')

print('\n--- Estadísticos descriptivos ---')
display(df[FEATURES + ['log_price']].describe().T.round(3))

---
## 8. Train/Test Split 80/20

Se realiza un split aleatorio estratificado con **SEED = 42** para garantizar reproducibilidad.

### Tamaños
- **Train (80%):** ~17.289 viviendas — usadas para entrenar y validar los modelos (cross-validation)
- **Test (20%):**  ~4.323 viviendas — reservadas para la evaluación final honesta

### Por qué guardamos `coords_train` / `coords_test` separadas
El notebook 03 construye los **retardos espaciales KNN** (media del precio de los K vecinos
más próximos). Para evitar **data leakage espacial**, los vecinos de cada vivienda de test
deben buscarse **solo entre las viviendas de train**. Si incluyéramos test en el árbol KNN,
estaríamos usando información del target de test para construir features de test.

Las coordenadas se exportan como arrays separados para facilitar esta lógica en el notebook 03.

In [ ]:
X = df[FEATURES]
y = df['log_price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)

# --- Guardar splits principales ---
X_train.to_csv(PROC_PATH + 'X_train.csv', index=False)
X_test.to_csv( PROC_PATH + 'X_test.csv',  index=False)
y_train.to_csv(PROC_PATH + 'y_train.csv', index=False, header=True)
y_test.to_csv( PROC_PATH + 'y_test.csv',  index=False, header=True)

# --- Coordenadas separadas para notebook 03 (KNN lags sin data leakage) ---
X_train[['lat', 'long']].to_csv(PROC_PATH + 'coords_train.csv', index=False)
X_test[['lat', 'long']].to_csv( PROC_PATH + 'coords_test.csv',  index=False)

# --- Dataset limpio completo (para SLX y análisis adicionales) ---
df.to_csv(PROC_PATH + 'kc_clean.csv', index=False)

print(f'Train : {X_train.shape[0]:,} viviendas | {X_train.shape[0]/len(df)*100:.1f}%')
print(f'Test  : {X_test.shape[0]:,}  viviendas | {X_test.shape[0]/len(df)*100:.1f}%')
print(f'\nArchivos guardados en {PROC_PATH}:')
for f in sorted(os.listdir(PROC_PATH)):
    path = PROC_PATH + f
    size_kb = os.path.getsize(path) / 1024
    print(f'  {f:<25} {size_kb:>8.1f} KB')

---
## 9. Resumen de transformaciones aplicadas

In [ ]:
resumen = pd.DataFrame({
    'Transformación': [
        'Eliminar id',
        'date → sale_month',
        'yr_built → house_age',
        'yr_renovated → renovated (0/1)',
        'Eliminar outlier bedrooms=33',
        'Eliminar sqft_basement',
        'price → log_price',
        'Train/Test split 80/20'
    ],
    'Justificación': [
        'Identificador sin valor predictivo',
        'Captura estacionalidad del mercado inmobiliario',
        'Antigüedad relativa al momento de la venta; más interpretable y portable',
        '95.8% sin renovar; t-test p=1.43e-63 valida que el hecho binario discrimina precio',
        'Error de entrada: 49 ft²/habitación (4.6 m²) — físicamente imposible',
        'Colinealidad perfecta: sqft_basement = sqft_living − sqft_above (100% casos)',
        'Skewness 4.02 → 0.43; error porcentual; reduce influencia de outliers de precio',
        'Evaluación honesta con datos no vistos; SEED=42 para reproducibilidad'
    ],
    'Variables resultantes': [
        '−1 columna',
        '+1 (sale_month) −1 (date)',
        '+1 (house_age) −1 (yr_built)',
        '+1 (renovated) −1 (yr_renovated)',
        '−1 registro (21.613→21.612)',
        '−1 columna',
        '+1 (log_price)',
        'Train 17.289 | Test 4.323'
    ]
})

display(resumen.style.set_properties(**{'text-align': 'left'}))

print('\n=== Estado final del dataset ===')
print(f'  Registros   : {len(df):,}')
print(f'  Features    : {len(FEATURES)}')
print(f'  Target      : log_price (np.log(price))')
print(f'  Train split : {len(X_train):,} viviendas')
print(f'  Test split  : {len(X_test):,} viviendas')
print(f'  Nulos       : {df[FEATURES + ["log_price"]].isnull().sum().sum()}')
print(f'  Skewness target: {y.skew():.3f}')